# Show-o 和离散扩散模型

Transfusion 混合了连续和离散表征。Show-o 选择了另外一条路：文本token使用因果的下一token预测，图像token使用MaskGIT的掩码离散扩散思想。都在同一个transformer中使用混合的注意力掩码。结果将VQA，文生图，图像修复以及多模态生成放在同一个骨架中，每个模态一个tokenzier，一个损失公式（下一token预测变成了掩码预测）。

## 问题描述

Transfusion 的双训练损失（文NTP，图MSE）能工作，但是需要一些动态的技巧————连续扩散的损失与NTP损失的数值量级不一样。平衡损失权重是一个操参数搜索过程。架构很高效也很复杂。

Show-o的答案是：保持模态离散（像Chameleon一样），不过在生成图像的时候使用掩码离散扩散而不是序列自回归。训练目标变成掩码token训练，能自然推广到下一token训练。

# 基本概念

## 掩码离散扩散

最初的MaskGIT技巧很优雅。从一个纯掩码图片开始（每个token都是特殊的`<MASK>`标记）。在每一步中，并行的预测所有被掩码的token，然后保留k个最自信的预测，并将其余的token重新掩码。经过～8-16轮迭代后，所有的token都被填入。每一步去掩码多少token的调度是经过了调优的————cosine调度工作效果最好。

训练起来很简单，从`[0-1]` 中采样一个掩码比例，然后应用到图像的VQ tokens，训练transformer学会修复被掩码的token。就像BERT对文本做的事情一样，被泛化到了图像生成。

## Show-o：一个Transformer，混合的掩码

Show-o 将MaskGIT 放入了因果语言模型transformer中，注意力掩码为：
- 文本token。 因果（标准LLM）
- 图像tokens。 图像块内全部双向（因为被掩码的tokens在预测时能够看见其其他图像token）
- 文生图。 图像能关注到之前的图像，图像能够关注到之前的文本。

训练过程是交替的：
- 对于文本序列使用标准的NTP。
- T2I 样本： 在被掩码的图像token上做生图，然后预测损失。
- VQA 样本： 在被掩码的图像token上生成，就跟常规NTP一样。

同一个损失是在`<MASK>` token 上算交叉熵损失，即能够覆盖文本NTP（只有最后一个token倍标记成“masked”）以及图像掩码扩散（随机子集被掩）

## 并行采样

Show-o 花大约16步生成一张图片，而不像自回归（约1000步），或者扩散（约20步）。每一步中，并行的预测所有被掩码的token，提交最自信的k的token，重复。

对比：
- Chameleon 和 Emu3 （自回归）：需要N词前向，每张图典型1024～4096步。
- Transfusion（连续扩散）：约20步，每步一次全量的transformer前向。
- Show-o（掩码离散扩散）：约16步，每步一次全量的transformer前向。

同等尺度模型下Show-o 比 Chameleon块，大概与Transfusion一致，因为Transfusion每步的开销更少（离散词表logits和连续MSE损失）。

## 同一个检查点，多个任务

Show-o 在推理时支持四种任务，通过提示词格式进行选取：
- 文本生成： 标准的文本自回归输出
- VAQ： 图像进，文本出
- T2I: 文本进，图像通过掩码离散扩散进行生产。
- 图像修复：带掩码的图像进，然后进行填充。

图像修复的能力是免费来自训练阶段中的掩码预测的。将VQ-token网格的一块区域进行掩码，然后把剩下的东西带上一段提示词喂给模型，预测被掩去的tokens。

## 掩码调度

推荐是cosine：
```
mask_ratio(t) = cos(pi * t/ (2 * T)) # t = 0..T
```
在第0步，所有的tokens都被掩码（比例为1.0）。在第T步，没有东西被掩码。余弦会把大部分权重集中在中程距离的比值上，因为在这个距离预测最有信息量。

## Show-o 使用的地方

2026年的分类：
- 离散token + NPT： Chameleon，Emu3， 简单但是推理慢。
- 离散token + 掩码扩散： Show-o， MaskGIT，LlamaGen，Muse。 并行采样，还是因为分词器，损失大。
- 连续 + 扩散： Transfusion，MMDiT，DiT。质量最高，训练起来最复杂。
- 连续 + 流匹配： JanusFlow， InternVL-U，最新。

通过任务需求进行选择：Transfusion，MMDiT，DiT。质量最高，训练起来最复杂。在质量要求很高，而且可以处理双损失管理时使用Transfusion。

# 开始编码

教学积木：Show-o / MaskGIT 核心——**cosine 掩码调度**、**因果文本 + 同图块双向掩码**、**统一「预测被掩离散 token」的 CE**（NTP 为其特例）、**并行置信度去掩采样**。


## 1. 配置、cosine 调度与随机掩码


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class TinyShowOConfig:
    """Show-o / 掩码离散扩散教学配置（远小于真模型）。"""

    text_vocab_size: int = 64
    """纯文本 BPE 词表大小示意。"""

    image_codebook_size: int = 32
    """图像 VQ 码本大小 K。"""

    num_image_tokens: int = 16
    """一张图的离散 token 数（真模型常 1024）。"""

    dim: int = 64
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.0
    """教学默认关 dropout。"""

    id_pad: int = 0
    id_bos: int = 1
    id_eos: int = 2
    id_img_start: int = 3
    id_img_end: int = 4
    """边界符占文本区间低 id。"""

    @property
    def id_mask(self) -> int:
        """共享词表中的 ``<MASK>`` id（接在图像码本之后）。"""
        return self.img_token_offset + self.image_codebook_size

    @property
    def img_token_offset(self) -> int:
        """图像码本在共享词表中的起始 id。"""
        return self.text_vocab_size

    @property
    def shared_vocab_size(self) -> int:
        """文本 + 图像码本 + ``<MASK>``。"""
        return self.text_vocab_size + self.image_codebook_size + 1


def cosine_mask_ratio(t: torch.Tensor, t_max: float = 1.0) -> torch.Tensor:
    """
    Cosine 掩码比例：``mask_ratio(t) = cos(π t / (2 T))``，``t∈[0,T]`` 时从 1→0。

    Args:
        t: 任意形状，取值建议在 ``[0, t_max]``。
        t_max: 调度终点 ``T``（训练时常取 1.0）。

    Returns:
        ratio: 与 ``t`` 同形状，取值约在 ``[0, 1]``。
    """
    x = (t / t_max).clamp(0.0, 1.0)
    return torch.cos(0.5 * torch.pi * x)


def sample_mask_ratio(batch_size: int, device: torch.device | None = None) -> torch.Tensor:
    """
    训练时为每个样本采样一个掩码比例（先采 ``u~U(0,1)``，再映射到 cosine）。

    Args:
        batch_size: batch 大小。
        device: 可选设备。

    Returns:
        ratio: ``(B,)``，约在 ``(0, 1]``。
    """
    u = torch.rand(batch_size, device=device)
    # u=0 → ratio≈1（几乎全掩）；u=1 → ratio≈0
    return cosine_mask_ratio(u, t_max=1.0).clamp_min(1e-6)


def apply_random_mask(
    token_ids: torch.Tensor,
    mask_id: int,
    ratio: torch.Tensor,
    eligible: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    按样本级比例随机把部分位置换成 ``mask_id``（MaskGIT 训练）。

    Args:
        token_ids: ``(B, L)`` 干净离散 id。
        mask_id: ``<MASK>`` 的共享词表 id。
        ratio: ``(B,)`` 每条样本的目标掩码比例。
        eligible: ``(B, L)`` 或 ``(L,)`` bool；``None`` 表示所有位置可掩。

    Returns:
        masked_ids: ``(B, L)``。
        mask: ``(B, L)`` bool，``True`` 表示该位被掩、计入 CE。
    """
    B, L = token_ids.shape
    device = token_ids.device
    if eligible is None:
        eligible = torch.ones(B, L, dtype=torch.bool, device=device)
    elif eligible.ndim == 1:
        eligible = eligible.view(1, L).expand(B, L)
    # 对每个样本：在 eligible 位置上按 Bernoulli(ratio) 掩码
    probs = ratio.view(B, 1).expand(B, L)
    rand = torch.rand(B, L, device=device)
    mask = (rand < probs) & eligible
    # 保证每条至少掩 1 个 eligible 位（避免空损失）
    for b in range(B):
        if eligible[b].any() and not mask[b].any():
            idx = torch.nonzero(eligible[b], as_tuple=False)[0, 0]
            mask[b, idx] = True
    masked_ids = token_ids.clone()
    masked_ids[mask] = mask_id
    return masked_ids, mask


def image_indices_to_shared_ids(indices: torch.Tensor, cfg: TinyShowOConfig) -> torch.Tensor:
    """
    Args:
        indices: ``(B, N)`` 码本下标 ``[0, K)``。
        cfg: 配置。

    Returns:
        ids: ``(B, N)`` 共享词表图像 id。
    """
    return indices + cfg.img_token_offset


def shared_ids_to_image_indices(ids: torch.Tensor, cfg: TinyShowOConfig) -> torch.Tensor:
    """
    Args:
        ids: ``(B, N)`` 共享词表图像 id。
        cfg: 配置。

    Returns:
        indices: ``(B, N)`` 码本下标。
    """
    return ids - cfg.img_token_offset


print(
    f"mask schedule ready | vocab={TinyShowOConfig().shared_vocab_size} "
    f"mask_id={TinyShowOConfig().id_mask}"
)


## 2. 混合注意力掩码 + 序列布局


In [ ]:
def build_showo_attn_mask(
    is_image: torch.Tensor,
    image_block_id: torch.Tensor,
) -> torch.Tensor:
    """
    Show-o 风格布尔允许掩码：``True`` = 允许注意。

    - 文本（含 ``<image>`` 边界符）：因果 ``j <= i``
    - 同图块图像 token：双向
    - 图像 → 更早文本；文本 → 更早图像：``j < i``

    Args:
        is_image: ``(L,)`` bool，连续图像 VQ token（不含边界符）。
        image_block_id: ``(L,)`` long，同块共享非负 id；非图像为 ``-1``。

    Returns:
        allow: ``(L, L)`` bool。
    """
    L = int(is_image.numel())
    device = is_image.device
    i = torch.arange(L, device=device).view(L, 1)
    j = torch.arange(L, device=device).view(1, L)
    text_i, text_j = ~is_image, ~is_image
    img_i, img_j = is_image, is_image
    causal_tt = text_i & text_j & (j <= i)
    same_block = (
        img_i
        & img_j
        & (image_block_id.view(L, 1) >= 0)
        & (image_block_id.view(L, 1) == image_block_id.view(1, L))
    )
    look_back = (j < i) & ((text_i & img_j) | (img_i & text_j) | (img_i & img_j))
    return causal_tt | same_block | look_back


def make_t2i_layout(
    cfg: TinyShowOConfig,
    prefix_len: int,
    suffix_len: int = 0,
) -> tuple[torch.Tensor, torch.Tensor, dict[str, slice]]:
    """
    预定义 T2I 布局：``prefix | <image> | N img tokens | </image> | suffix``。

    Args:
        cfg: 配置。
        prefix_len: 前缀文本长度。
        suffix_len: 后缀文本长度。

    Returns:
        is_image: ``(L,)``。
        image_block_id: ``(L,)``。
        spans: 各段 ``slice``。
    """
    n = cfg.num_image_tokens
    L = prefix_len + 1 + n + 1 + suffix_len
    is_image = torch.zeros(L, dtype=torch.bool)
    block = torch.full((L,), -1, dtype=torch.long)
    p0 = prefix_len + 1
    p1 = p0 + n
    is_image[p0:p1] = True
    block[p0:p1] = 0
    spans = {
        "prefix": slice(0, prefix_len),
        "img_start": slice(prefix_len, prefix_len + 1),
        "image": slice(p0, p1),
        "img_end": slice(p1, p1 + 1),
        "suffix": slice(p1 + 1, L),
    }
    return is_image, block, spans


def pack_t2i_sequence(
    text_prefix: torch.Tensor,
    image_indices: torch.Tensor,
    cfg: TinyShowOConfig,
    text_suffix: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, dict[str, slice]]:
    """
    打包干净 T2I 序列（图像为 VQ 索引，稍后可再掩码）。

    Args:
        text_prefix: ``(B, Lp)``。
        image_indices: ``(B, N)`` 码本下标。
        cfg: 配置。
        text_suffix: ``(B, Ls)`` 或 ``None``。

    Returns:
        text_ids: ``(B, L)`` 共享词表 id（干净）。
        is_image: ``(L,)``。
        image_block_id: ``(L,)``。
        spans: 槽位。
    """
    B = text_prefix.size(0)
    if text_suffix is None:
        text_suffix = text_prefix.new_zeros(B, 0)
    is_image, block, spans = make_t2i_layout(cfg, text_prefix.size(1), text_suffix.size(1))
    start = torch.full((B, 1), cfg.id_img_start, device=text_prefix.device, dtype=torch.long)
    end = torch.full((B, 1), cfg.id_img_end, device=text_prefix.device, dtype=torch.long)
    img_ids = image_indices_to_shared_ids(image_indices, cfg)
    text_ids = torch.cat([text_prefix, start, img_ids, end, text_suffix], dim=1)
    return text_ids, is_image.to(text_prefix.device), block.to(text_prefix.device), spans


print("Show-o attn mask + pack ready")


## 3. 双模态 Transformer：在 ``<MASK>`` 上统一 CE


In [ ]:
class MaskedSelfAttention(nn.Module):
    """外部布尔允许掩码的多头自注意力。"""

    def __init__(self, dim: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        if dim % n_heads:
            raise ValueError("dim must divide n_heads")
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.out = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, allow: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, L, D)``。
            allow: ``(L, L)`` 或 ``(B, L, L)``，``True``=允许。

        Returns:
            y: ``(B, L, D)``。
        """
        B, L, D = x.shape
        qkv = self.qkv(x).view(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim**-0.5)
        mask = allow.view(1, 1, L, L) if allow.ndim == 2 else allow.unsqueeze(1)
        attn = attn.masked_fill(~mask, float("-inf"))
        attn = self.drop(attn.softmax(dim=-1))
        h = (attn @ v).transpose(1, 2).reshape(B, L, D)
        return self.out(h)


class ShowOBlock(nn.Module):
    """Pre-LN Transformer 块。"""

    def __init__(self, dim: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MaskedSelfAttention(dim, n_heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, x: torch.Tensor, allow: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, L, D)``。
            allow: 注意力允许掩码。

        Returns:
            y: ``(B, L, D)``。
        """
        x = x + self.attn(self.norm1(x), allow)
        x = x + self.mlp(self.norm2(x))
        return x


class TinyShowO(nn.Module):
    """共享词表 Transformer：只在被掩位置上算 CE（统一图像掩码扩散与文本 NTP 特例）。"""

    def __init__(self, cfg: TinyShowOConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg.shared_vocab_size, cfg.dim)
        self.pos = nn.Embedding(512, cfg.dim)
        self.blocks = nn.ModuleList(
            [ShowOBlock(cfg.dim, cfg.n_heads, cfg.dropout) for _ in range(cfg.n_layers)]
        )
        self.norm = nn.LayerNorm(cfg.dim)
        self.head = nn.Linear(cfg.dim, cfg.shared_vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor, allow: torch.Tensor) -> torch.Tensor:
        """
        Args:
            input_ids: ``(B, L)`` 共享词表 id（可含 ``<MASK>``）。
            allow: ``(L, L)`` 注意力掩码。

        Returns:
            logits: ``(B, L, V)``。
        """
        B, L = input_ids.shape
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        x = self.tok(input_ids) + self.pos(pos)
        for blk in self.blocks:
            x = blk(x, allow)
        return self.head(self.norm(x))

    def masked_ce_loss(
        self,
        input_ids: torch.Tensor,
        clean_ids: torch.Tensor,
        loss_mask: torch.Tensor,
        allow: torch.Tensor,
    ) -> torch.Tensor:
        """
        仅在 ``loss_mask==True`` 的位置，用当前隐状态预测 **该位** 的干净 token。

        Args:
            input_ids: ``(B, L)`` 已掩输入。
            clean_ids: ``(B, L)`` 干净标签。
            loss_mask: ``(B, L)`` bool。
            allow: ``(L, L)``。

        Returns:
            loss: 标量交叉熵。
        """
        logits = self.forward(input_ids, allow)
        if not loss_mask.any():
            return logits.sum() * 0.0
        return F.cross_entropy(logits[loss_mask], clean_ids[loss_mask])


def ntp_as_mask_targets(
    clean_ids: torch.Tensor,
    cfg: TinyShowOConfig,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    把因果 NTP 写成「只掩下一个位置」的掩码预测特例。

    对长度 ``L`` 的纯文本：输入为 ``ids[:, :-1]`` 可见，在末尾追加一个 ``<MASK>``，
    只对该 mask 位预测 ``ids[:, -1]``。教学用单步示意。

    Args:
        clean_ids: ``(B, L)`` 纯文本（``L>=2``）。
        cfg: 配置。

    Returns:
        masked_ids: ``(B, L)``，最后一位为 ``<MASK>``，前面为 ``clean[:, :-1]`` 再 pad 对齐。
        loss_mask: ``(B, L)``，仅最后一位为 True。
    """
    B, L = clean_ids.shape
    if L < 2:
        raise ValueError("need L>=2")
    masked = clean_ids.clone()
    masked[:, -1] = cfg.id_mask
    loss_mask = torch.zeros(B, L, dtype=torch.bool, device=clean_ids.device)
    loss_mask[:, -1] = True
    return masked, loss_mask


print("TinyShowO + unified masked CE ready")


## 4. 并行采样：按置信度逐步去掩（MaskGIT）


In [ ]:
def num_to_unmask(still_masked: int, step: int, num_steps: int) -> int:
    """
    根据 cosine 调度决定本步应新揭开多少个 token。

    Args:
        still_masked: 当前仍被掩的个数。
        step: ``0 .. num_steps-1``。
        num_steps: 总步数 ``T``。

    Returns:
        k: 本步至少揭开 1 个（若还有剩余）。
    """
    if still_masked <= 0:
        return 0
    # 目标：步 step 结束后剩余掩码比例 = cos(π (step+1) / (2T))
    t_now = float(step) / num_steps
    t_next = float(step + 1) / num_steps
    r_now = float(torch.cos(torch.tensor(0.5 * torch.pi * t_now)))
    r_next = float(torch.cos(torch.tensor(0.5 * torch.pi * t_next)))
    # still_masked 对应比例 r_now 的绝对数量；要降到 r_next
    # 用相对差额估算揭开数
    frac = max(0.0, (r_now - r_next) / max(r_now, 1e-8))
    k = int(round(still_masked * frac))
    return max(1, min(still_masked, k))


@torch.no_grad()
def maskgit_sample_image(
    model: TinyShowO,
    text_prefix: torch.Tensor,
    num_steps: int = 8,
) -> torch.Tensor:
    """
    文生图：图像槽初始全 ``<MASK>``，每步并行预测，提交置信度最高的 k 个。

    Args:
        model: ``TinyShowO``。
        text_prefix: ``(B, Lp)``。
        num_steps: 去掩迭代步数（笔记约 8–16）。

    Returns:
        image_indices: ``(B, N)`` 码本下标。
    """
    cfg = model.cfg
    B = text_prefix.size(0)
    device = text_prefix.device
    fake_idx = torch.zeros(B, cfg.num_image_tokens, device=device, dtype=torch.long)
    clean, is_image, block, spans = pack_t2i_sequence(text_prefix, fake_idx, cfg)
    allow = build_showo_attn_mask(is_image, block)
    ids = clean.clone()
    ids[:, spans["image"]] = cfg.id_mask
    img_slice = spans["image"]

    for step in range(num_steps):
        logits = model(ids, allow)  # (B, L, V)
        # 只在图像码本区间上取预测（不含 MASK/文本）
        img_logits = logits[:, img_slice, cfg.img_token_offset : cfg.id_mask]
        probs = img_logits.softmax(dim=-1)
        conf, pred_local = probs.max(dim=-1)  # (B, N)
        pred_ids = pred_local + cfg.img_token_offset

        still = ids[:, img_slice] == cfg.id_mask  # (B, N)
        for b in range(B):
            m = still[b]
            n_left = int(m.sum().item())
            if n_left == 0:
                continue
            k = num_to_unmask(n_left, step, num_steps)
            # 只在仍掩位置上比置信度
            score = conf[b].clone()
            score[~m] = -1.0
            topk = torch.topk(score, k=k).indices
            ids[b, img_slice.start + topk] = pred_ids[b, topk]

    return shared_ids_to_image_indices(ids[:, img_slice], cfg)


@torch.no_grad()
def inpaint_image_tokens(
    model: TinyShowO,
    text_prefix: torch.Tensor,
    known_indices: torch.Tensor,
    hole_mask: torch.Tensor,
    num_steps: int = 8,
) -> torch.Tensor:
    """
    图像修复：已知 VQ token 保留，``hole_mask`` 位置从 ``<MASK>`` 并行填回。

    Args:
        model: 模型。
        text_prefix: ``(B, Lp)``。
        known_indices: ``(B, N)`` 完整网格（洞内值可任意，会被盖掉）。
        hole_mask: ``(B, N)`` 或 ``(N,)``，``True``=需要修复。
        num_steps: 去掩步数。

    Returns:
        filled_indices: ``(B, N)``。
    """
    cfg = model.cfg
    B = text_prefix.size(0)
    if hole_mask.ndim == 1:
        hole_mask = hole_mask.view(1, -1).expand(B, -1)
    clean, is_image, block, spans = pack_t2i_sequence(text_prefix, known_indices, cfg)
    allow = build_showo_attn_mask(is_image, block)
    ids = clean.clone()
    img = spans["image"]
    ids[:, img][hole_mask] = cfg.id_mask

    for step in range(num_steps):
        logits = model(ids, allow)
        img_logits = logits[:, img, cfg.img_token_offset : cfg.id_mask]
        probs = img_logits.softmax(dim=-1)
        conf, pred_local = probs.max(dim=-1)
        pred_ids = pred_local + cfg.img_token_offset
        still = ids[:, img] == cfg.id_mask
        for b in range(B):
            m = still[b]
            n_left = int(m.sum().item())
            if n_left == 0:
                continue
            k = num_to_unmask(n_left, step, num_steps)
            score = conf[b].clone()
            score[~m] = -1.0
            topk = torch.topk(score, k=min(k, n_left)).indices
            ids[b, img.start + topk] = pred_ids[b, topk]
    return shared_ids_to_image_indices(ids[:, img], cfg)


print("MaskGIT parallel sample + inpaint ready")


## 5. 冒烟测试


In [ ]:
def smoke_test() -> None:
    """验证掩码调度、统一 CE、并行去掩与修复。"""
    torch.manual_seed(0)
    cfg = TinyShowOConfig()
    model = TinyShowO(cfg)

    print("=== cosine schedule ===")
    ts = torch.tensor([0.0, 0.5, 1.0])
    rs = cosine_mask_ratio(ts)
    print(f"t={ts.tolist()} ratio={[round(float(x), 4) for x in rs]}")
    assert abs(float(rs[0]) - 1.0) < 1e-5
    assert float(rs[-1]) < 1e-5

    print("\n=== attn mask ===")
    is_image, block, spans = make_t2i_layout(cfg, prefix_len=3, suffix_len=1)
    allow = build_showo_attn_mask(is_image, block)
    ps = spans["image"]
    assert bool(allow[ps.start, ps.start + 1].item())  # 图像双向
    assert not bool(allow[0, ps.start].item())  # 前缀看不到未来图像
    print(f"allow={tuple(allow.shape)} OK")

    print("\n=== T2I masked CE ===")
    B = 2
    prefix = torch.randint(5, cfg.text_vocab_size, (B, 4))
    img_idx = torch.randint(0, cfg.image_codebook_size, (B, cfg.num_image_tokens))
    clean, is_image, block, spans = pack_t2i_sequence(prefix, img_idx, cfg)
    allow = build_showo_attn_mask(is_image, block)
    ratio = sample_mask_ratio(B)
    masked, loss_mask = apply_random_mask(
        clean, cfg.id_mask, ratio, eligible=is_image
    )
    # 文本位不掩；损失只在图像被掩处
    assert not loss_mask[:, ~is_image].any()
    loss = model.masked_ce_loss(masked, clean, loss_mask, allow)
    print(f"loss={loss.item():.4f} masked_frac={loss_mask.float().mean().item():.3f}")
    loss.backward()
    assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in model.parameters())

    print("\n=== NTP as mask special case ===")
    text = torch.randint(5, cfg.text_vocab_size, (2, 6))
    m_ids, m_loss = ntp_as_mask_targets(text, cfg)
    L = text.size(1)
    allow_txt = torch.tril(torch.ones(L, L, dtype=torch.bool))
    ntp_loss = model.masked_ce_loss(m_ids, text, m_loss, allow_txt)
    print(f"ntp_special_loss={ntp_loss.item():.4f}")

    print("\n=== parallel MaskGIT sample ===")
    model.eval()
    out = maskgit_sample_image(model, prefix[:1], num_steps=8)
    print(f"sampled_indices={tuple(out.shape)}")
    assert out.shape == (1, cfg.num_image_tokens)
    assert int(out.min()) >= 0 and int(out.max()) < cfg.image_codebook_size

    print("\n=== inpaint ===")
    known = torch.randint(0, cfg.image_codebook_size, (1, cfg.num_image_tokens))
    hole = torch.zeros(cfg.num_image_tokens, dtype=torch.bool)
    hole[2:6] = True
    filled = inpaint_image_tokens(model, prefix[:1], known, hole, num_steps=6)
    assert torch.equal(filled[0, ~hole], known[0, ~hole])
    print(f"filled={tuple(filled.shape)} known_kept=OK")

    print("SMOKE TEST OK")


smoke_test()
